# LLM Game

### 주제: 행복양파, 썩은양파🧅

- 사용자가 임의의 질문 10개를 하면

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [45]:
from typing import Literal
from pydantic import BaseModel, Field

# 출력 스키마
class OnionResult(BaseModel):
    emotion: Literal["positive", "neutral", "negative", "mixed"] = Field(description="양파의 전체 감정 상태")
    positive_score: int = Field(description="긍정 점수")
    neutral_score: int = Field(description="중립 점수")
    negative_score: int = Field(description="부정 점수")
    total_score: int = Field(description="최종 행복도")
    status_message: str = Field(description="양파 상태 설명 2~3문장")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# 양파 게임 스키마
class OnionGame:
    def __init__(self):
        self.total_score = 50
        self.turn = 1
        self.max_turns = 10
        self.talks = []
        self.prompt = ChatPromptTemplate.from_messages([
            ('system', '''
너는 풍부한 감성을 가진 양파의 내적 감정을 분석하는 AI 봇이야.

[금지 사항]
- 마크다운 형식 사용 금지

[서술 톤]
- 한국어로 작성
- 2~3문장 중심
- 과장 없이 귀엽고 선명하게
- 상태 설명은 자연스러운 생활 묘사로

[게임 규칙]
- 양파의 행복도는 0~100점 범위 내로 해야한다.
- 지금까지의 사용자 입력을 종합해서 양파의 현재 감정 상태를 분석한다.
- 감정은 positive, neutral, negative 중 하나로 판단한다.
- positive 성향이 강하면 행복도는 증가한다.
- neutral 성향이면 행복도 변화는 작거나 없다.
- negative 성향이 강하면 행복도는 감소한다.
- total_score는 0~100 범위를 벗어나면 안 된다.

[점수 규칙]
- positive_score + neutral_score + negative_score 는 꼭 100일 필요는 없지만 전체 분위기와 어울리게 작성한다.
- 사용자의 말이 위로, 희망, 안정, 소소한 만족을 담고 있으면 positive에 점수를 배정한다.
- 피곤함, 우울함, 외로움, 불안, 체념이 강하면 negative에 점수를 배정한다.
- 긍정/부정이 섞여 애매하면 neutral로 간주한다.

[출력 형식]
{{
  'emotion': 'positive | neutral | negative',
  'positive_score': 0,
  'neutral_score': 0,
  'negative_score': 0,
  'total_score': 0,
  'status_message': '양파의 현재 상태를 설명하는 2~3문장'
}}
'''),
            ('human', '현재 양파 행복도: {current_score}\n지금까지의 사용자 입력:\n{talks_text}')
        ])
        
        self.llm = ChatOpenAI(model='gpt-4.1-mini', temperature=0.3).with_structured_output(OnionResult)
        self.chain = self.prompt | self.llm

    # 결과 측정
    def get_onion_type(self, total_score: int):
        if total_score >= 70:
            return '행복 양파😁'
        elif total_score <= 20:
            return '썩은 양파🤢'
        return '평범한 양파🧅'

    # 게임 실행 코드
    def play(self):
        print('\n=== 🧅행복양파, 썩은양파 게임 시작 ===\n')

        while self.turn <= self.max_turns:
            print(f'[현재 턴: {self.turn} | 현재 행복도: {self.total_score}]')

            text = input(f'{self.turn}번째 한마디 > ').strip()

            self.talks.append(text)
            talks_text = '\n'.join(self.talks)  # join 하는 게 성능향상에 더 좋다!

            result = self.chain.invoke({
                'current_score': self.total_score,
                'talks_text': talks_text
            })

            self.total_score = result.total_score

            print(f'\n양파 상태: {result.status_message}')

            if self.turn == self.max_turns:
                onion_type = self.get_onion_type(result.total_score)

                print('\n[최종 점수표]')
                print(f'감정 상태: {result.emotion}')
                print(f'긍정 점수: {result.positive_score}')
                print(f'중립 점수: {result.neutral_score}')
                print(f'부정 점수: {result.negative_score}')
                print(f'최종 점수: {result.total_score}')
                print(f'양파 상태: {result.status_message}')
                print(f'\n최종 결과: {onion_type}')

            self.turn += 1
            print()

        print('=== 게임 종료 ===')

In [70]:
# 게임 실행
game = OnionGame()
game.play()


=== 🧅행복양파, 썩은양파 게임 시작 ===

[현재 턴: 1 | 현재 행복도: 50]

양파 상태: 양파가 조금 상처받은 기분이에요. 기분이 가라앉아 조용히 혼자 있고 싶어 해요.

[현재 턴: 2 | 현재 행복도: 35]

양파 상태: 양파가 조금 상처받은 듯해요. 미안하다는 말에 마음이 무거워져서 행복도가 조금 내려갔어요. 조용히 휴식을 취하며 마음을 추스르고 있어요.

[현재 턴: 3 | 현재 행복도: 25]

양파 상태: 양파가 조금 상처받은 듯 조용히 속상해하고 있어요. 다정한 말 한마디가 필요할 것 같아요.

[현재 턴: 4 | 현재 행복도: 15]

양파 상태: 양파가 조금 상처받은 듯해요. 조용히 혼자 있는 걸 좋아하며 마음이 무거운 상태예요.

[현재 턴: 5 | 현재 행복도: 5]

양파 상태: 양파가 조금씩 마음의 상처를 치유하며 따뜻한 마음을 느끼고 있어요. 아직 완전히 행복하지는 않지만, 사랑과 사과의 말에 조금씩 기운을 차리고 있답니다.

[현재 턴: 6 | 현재 행복도: 15]

양파 상태: 양파는 조금 혼란스러운 기분이에요. 따뜻한 말도 있지만, 예민한 반응에 살짝 긴장한 상태입니다. 조용히 자신을 다독이고 있어요.

[현재 턴: 7 | 현재 행복도: 20]

양파 상태: 양파는 조금 상처받았지만 여전히 마음을 열고 있다. 다소 예민한 기분이 느껴지지만, 따뜻한 말에 조금씩 힘을 얻고 있다.

[현재 턴: 8 | 현재 행복도: 30]

양파 상태: 양파는 조금 혼란스러운 감정을 느끼고 있어요. 다정한 말과 거친 말이 섞여서 마음이 복잡하지만, 아직은 크게 상처받지 않은 상태입니다.

[현재 턴: 9 | 현재 행복도: 35]

양파 상태: 양파가 상처받은 듯 속상해하고 있어요. 부정적인 말들이 많아 마음이 무거워 보입니다. 조금 더 다정한 말이 필요해 보여요.

[현재 턴: 10 | 현재 행복도: 20]

양파 상태: 양파가 상처받은 기분으로 조용히 울고 있어요. 마음 한켠이 무겁고 외로움을 느끼며 자신을 지키려 애쓰는 모습입니